# TP 4.1 — Analyse exploratoire (IVAL + jointure annuaire pour l’EP)**Données** :- IVAL GT : `donnees/fr-en-indicateurs-de-resultat-des-lycees-gt_v2.csv`- Annuaire (priorité / EP) : `donnees/fr-en-annuaire-education.csv`- Optionnel IPS : `donnees/fr-en-ips-lycees-ap2023.csv`Voir `Docs/sources_data.md` section TP 4.1.

In [ ]:
from pathlib import Path

def resolve_data_dir() -> Path:
    """Racine du dépôt = dossier qui contient `donnees/`. Lancer Jupyter depuis cette racine."""
    cwd = Path.cwd()
    if (cwd / "donnees").is_dir():
        return cwd
    p = cwd
    for _ in range(4):
        if (p / "donnees").is_dir():
            return p
        p = p.parent
    return cwd

ROOT = resolve_data_dir()
DATA = ROOT / "donnees"
print("Répertoire données :", DATA.resolve())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ival_path = DATA / "fr-en-indicateurs-de-resultat-des-lycees-gt_v2.csv"
ann_path = DATA / "fr-en-annuaire-education.csv"
if not ival_path.exists() or not ann_path.exists():
    raise FileNotFoundError("Placez les CSV IVAL et Annuaire dans donnees/ (voir donnees/README.md)")

ival = pd.read_csv(ival_path, sep=";", low_memory=False)
ann = pd.read_csv(ann_path, sep=";", low_memory=False)
print(ival.shape, ann.shape)
ival.head(2)


In [ ]:
# Harmoniser types numériques (virgule décimale)
for c in ["taux_reu_total", "va_reu_total", "taux_acces_2nde", "taux_men_total"]:
    if c in ival.columns:
        ival[c] = pd.to_numeric(ival[c].astype(str).str.replace(",", "."), errors="coerce")

ival[["uai", "annee", "secteur", "libelle_academie", "taux_reu_total", "va_reu_total"]].describe(include="all")


In [ ]:
# Jointure avec l'annuaire pour disposer du flag EP (REP / REP+)
cols_ann = [
    "identifiant_de_l_etablissement",
    "appartenance_education_prioritaire",
]
sub = ann[cols_ann].drop_duplicates(subset=["identifiant_de_l_etablissement"])
merged = ival.merge(
    sub,
    left_on="uai",
    right_on="identifiant_de_l_etablissement",
    how="left",
)
merged["zone_ep"] = merged["appartenance_education_prioritaire"].fillna("Hors EP").replace(
    {"": "Hors EP"}
)
merged[["zone_ep"]].value_counts(dropna=False).head(10)


In [ ]:
# Statistiques par académie, secteur, zone EP
g = (
    merged.groupby(["libelle_academie", "secteur", "zone_ep"], dropna=False)
    .agg(
        taux_reu_median=("taux_reu_total", "median"),
        taux_reu_mean=("taux_reu_total", "mean"),
        n=("uai", "count"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
g.head(12)


In [ ]:
# Histogramme des valeurs ajoutées (réussite)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(merged["va_reu_total"].dropna(), kde=True, ax=ax)
ax.set_title("Distribution de la valeur ajoutée (réussite) — IVAL GT")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot par académie (limiter aux académies les plus représentées pour lisibilité)
top_acad = merged["libelle_academie"].value_counts().head(8).index
subp = merged[merged["libelle_academie"].isin(top_acad)]
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=subp, x="libelle_academie", y="va_reu_total", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_title("Valeur ajoutée (réussite) par académie (échantillon des 8 académies les plus fournies)")
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap de corrélation entre quelques taux
cols = [c for c in ["taux_reu_total", "taux_acces_2nde", "taux_men_total", "va_reu_total", "va_acces_2nde", "va_men_total"] if c in merged.columns]
corr = merged[cols].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Corrélations (taux / VA)")
plt.tight_layout()
plt.show()


In [ ]:
# Comparaison public / privé sous contrat
fig, ax = plt.subplots(figsize=(6, 4))
sns.violinplot(data=merged, x="secteur", y="taux_reu_total", ax=ax)
ax.set_title("Taux brut de réussite par secteur")
plt.tight_layout()
plt.show()


## Interprétation (10 lignes max)1. …2. …3. …**Limites** : effets de structure, seuils de publication DEPP, corrélation ≠ causalité, lecture prudente des comparaisons « valeur ajoutée ».